In [ ]:
import azure.ai.ml
print(azure.ai.ml.__file__)

# Connect to workspace

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

In [ ]:
ml_client = MLClient.from_config(credential=credential)

# Data exploration

In [ ]:
import pandas as pd
data_asset = ml_client.data.get(name="diabetes-lab-results-raw", version="e96149a2c898")

In [ ]:
from azure.identity import DefaultAzureCredential

datastore = ml_client.datastores.get_default()
blob_path = data_asset.path.split("/paths/")[-1]

df = pd.read_csv(
    f"abfs://{datastore.container_name}/{blob_path}",
    storage_options={"account_name": datastore.account_name, "credential": DefaultAzureCredential()},
)
df.head()

# Prepare data

In [ ]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

training_data = Input(type=AssetTypes.MLTABLE, path="azureml:diabetes-lab-results-raw:e96149a2c898")

# Configure automated machine learning job

- compute cluster: aml-cluster
- target column: diabetic (bool)
- primary metric: accuracy
- early stop: time limit (1h)
- cross validation: 5 models
- don't consider LogisticRegression

In [ ]:
from azure.ai.ml import automl

# configure de classification job
classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="auto-ml-class-dev",
    training_data=training_data,
    target_column_name="Diabetic",
    primary_metric="accuracy",
    n_cross_validations=5,
    enable_model_explainability=True,
)

# set limits
classification_job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True,
)

# set the trainig properties
classification_job.set_training(
    blocked_training_algorithms=["LogisticRegression"],
    enable_onnx_compatible_models=True,
)

# Run an automated machine learning job

In [ ]:
returned_job = ml_client.jobs.create_or_update(
    classification_job
)

aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)